In [4]:
 """
    Transforme la matrice Cq en une forme triangulaire inférieure,
    tout en enregistrant les indices des colonnes indépendantes et
    les coefficients des combinaisons linéaires dans LC.

    Paramètres :
    - Cq : numpy array, la matrice de chiffrement.
    - m : int, nombre de lignes de la matrice.
    - n : int, nombre de colonnes de la matrice.

    Retourne :
    - Cq : numpy array, la matrice triangulée.
    - LC : numpy array, les coefficients des combinaisons linéaires.
    """
import numpy as np

def lower_triangularize(Cq, m, n):
   
    # Initialisation de la matrice identité LC
    LC = np.eye(m, dtype=float)

    for k in range(n):  # Parcours des colonnes
        # Trouver le premier élément non nul dans la colonne k
        j = next((i for i in range(k, n) if Cq[k, i] != 0), None)
        
        if j is not None and Cq[k, j] != 0:
            # Diviser la colonne j par l'élément Cq[k, j]
            Cq[:, j] /= Cq[k, j]
            LC[:, j] /= Cq[k, j]
            
            # Échanger les colonnes j et k dans Cq et LC
            Cq[:, [j, k]] = Cq[:, [k, j]]
            LC[:, [j, k]] = LC[:, [k, j]]

            # Soustraire les multiples de la colonne pivot pour rendre les
            # éléments au-dessous de Cq[k, k] nuls
            for i in range(k + 1, m):
                factor = Cq[i, k]
                Cq[i, :] -= factor * Cq[k, :]
                LC[i, :] -= factor * LC[k, :]

    return Cq, LC


In [5]:
# Exemple de matrice Cq
Cq = np.array([
    [2, 4, 1],
    [4, 8, 3],
    [6, 9, 5]
], dtype=float)

m, n = Cq.shape
Cq, LC = lower_triangularize(Cq, m, n)

print("Matrice triangulée Cq :\n", Cq)
print("Matrice LC (combinaisons linéaires) :\n", LC)


Matrice triangulée Cq :
 [[ 1.          1.         -1.33333333]
 [ 0.          1.         -0.        ]
 [ 0.          0.          1.        ]]
Matrice LC (combinaisons linéaires) :
 [[ 1.  0.  0.]
 [-2.  0.  1.]
 [ 1.  1. -2.]]


In [7]:
 """
    Extrait les indices des colonnes indépendantes et les coefficients
    des combinaisons linéaires.

    Paramètres :
    - LC : numpy array, matrice des combinaisons linéaires.
    - Cq : numpy array, matrice triangulée inférieure.

    Retourne :
    - result : dict, contenant les indices des colonnes indépendantes
      et les coefficients des combinaisons linéaires.
    """


def get_linear_combinations(LC, Cq):
   
    size = 0
    ind = []  # Indices des colonnes libres
    result = {}  # Dictionnaire pour stocker les résultats
    
    # Première boucle : déterminer le nombre de colonnes indépendantes
    for i in range(Cq.shape[0]):  # Parcourt les lignes
        if Cq[i, i] != 0:
            size += 1

    # Deuxième boucle : récupérer les indices des colonnes indépendantes
    for i in range(LC.shape[0]):  # Parcourt les lignes de LC
        if len(ind) == size:
            break
        for j in range(size):  # Vérifie les colonnes jusqu'à "size"
            if LC[i, j] != 0:
                ind.append(i)
                break

    # Dernière boucle : extraire les coefficients des combinaisons linéaires
    for col in range(len(ind), LC.shape[1]):  # Parcourt les colonnes restantes
        for line in range(LC.shape[0]):  # Parcourt les lignes
            lineOK = True
            for k in ind:  # Vérifie si la ligne est déjà marquée
                if k == line:
                    lineOK = False
                    break
            if lineOK and LC[line, col] != 0:
                if 0 not in result:
                    result[0] = {}
                result[0][col - len(ind)] = line
                break

    # Remplir les résultats avec les coefficients
    for i, index in enumerate(ind):
        for col in range(len(ind), LC.shape[1]):
            if i + 1 not in result:
                result[i + 1] = {}
            result[i + 1][col - len(ind)] = -LC[index, col]

    return result


In [9]:
 """
    Reconstruit les blocs linéairement dépendants en fonction des combinaisons
    linéaires stockées dans LC.

    Paramètres :
    - res : numpy array, matrice initialisée aléatoirement.
    - LC : numpy array, matrice contenant les combinaisons linéaires.
    - ind : list, indices des colonnes indépendantes.
    - decrypt : fonction pour décrypter le résultat final.

    Retourne :
    - res : numpy array, matrice avec les blocs linéairement dépendants reconstruits.
    """

def rebuild_linearly_dependent_blocks(res, LC, ind, decrypt):
   
    for c in range(LC.shape[1]):  # Parcours des colonnes de LC
        col = LC[0, c]  # Colonne cible (dépendante)
        for line in range(LC.shape[0]):  # Parcours des lignes
            res[line, col] = 0  # Initialisation du bloc dépendant à 0
            for k in range(len(ind)):  # Combinaison linéaire avec les colonnes indépendantes
                res[line, col] += res[line, ind[k]] * LC[k + 1, c]

    # Appliquer la fonction de déchiffrement sur la matrice résultante
    return decrypt(res)


In [10]:
 """
    Implémente l'attaque basée sur les orbites pour déchiffrer le texte chiffré.

    Paramètres :
    - cipher : numpy array, le texte chiffré à déchiffrer.
    - p : int, taille de l'alphabet.
    - Algorithm_1 : fonction pour trianguler la matrice et générer LC.
    - Algorithm_2 : fonction pour récupérer les colonnes indépendantes.
    - Algorithm_3 : fonction pour reconstruire les blocs dépendants.
    - findWord : fonction pour vérifier si un mot clair est trouvé.

    Retourne :
    - guessMessage : numpy array, le texte clair correspondant.
    """


import numpy as np
import random

def orbit_based_attack(cipher, p, Algorithm_1, Algorithm_2, Algorithm_3, findWord):
   
    # Étape 1 : Triangularisation et obtention de LC
    in_matrix, LC = Algorithm_1(cipher, np.eye(cipher.shape[0]))
    
    # Étape 2 : Identification des colonnes indépendantes
    ind = Algorithm_2(LC, in_matrix)
    
    # Étape 3 : Générer une matrice de test aléatoire
    guess = np.zeros_like(cipher, dtype=float)
    for c in range(LC.shape[1]):  # Parcourt les colonnes de LC
        for l in range(LC.shape[0]):  # Parcourt les lignes
            guess[l, c] = random.randint(0, p - 1)  # Valeurs aléatoires dans l'alphabet

    # Étape 4 : Reconstruire le message basé sur les combinaisons
    guessMessage = Algorithm_3(guess, LC)

    # Étape 5 : Itération jusqu'à trouver le bon message
    while not findWord(guessMessage):
        # Incrémentation dans une logique d'odomètre
        guess += 1  # Ajout d'un pas arbitraire
        guess %= p  # Boucle dans l'alphabet pour chaque élément
        guessMessage = Algorithm_3(guess, LC)

    # Retourner le texte clair trouvé
    return guessMessage
